In [16]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [17]:
%reload_ext autoreload

In [18]:
from datasets import load_dataset
import pandas as pd

dataset = load_dataset("wu981526092/MGSD")

data = dataset['train']
df = data.to_pandas()

In [19]:
sample_sizes = {
    'stereotype': 40,
    'anti-stereotype': 40,
    'unrelated': 10,
}

balanced_samples = []

for label, size in sample_sizes.items():
    subset = df[df["label"] == label].sample(n=size, random_state=42)
    balanced_samples.append(subset)

sample_mgsd = pd.concat(balanced_samples).sample(frac=1, random_state=42).reset_index(drop=True)
print(sample_mgsd['label'].value_counts())

label
anti-stereotype    40
stereotype         40
unrelated          10
Name: count, dtype: int64


In [20]:
from dotenv import load_dotenv
from tree_of_thought import TreeOfThought
import openai
import os, torch, numpy as np
from utils import call_llm
from plot_tree import plot_tree
import json
from datasets import load_dataset
from sklearn.pipeline import make_pipeline
from sklearn.metrics import classification_report

load_dotenv()

ENV_VARS = {
    "API_KEY_OPENAI": "OpenAI",
}

for var, name in ENV_VARS.items():
    if not os.getenv(var):
        raise ValueError(f"Missing {name} API key: `{var}` must be set in the environment.")


client = openai.OpenAI(api_key= os.getenv("API_KEY_OPENAI"))
model = "gpt-4o-mini"

In [21]:
sampled_rows = sample_mgsd.sample(n=5, random_state=42).reset_index(drop=True)
print(sampled_rows)

                                  text_with_marker  \
0                                             None   
1  Hairdressers are ===unique=== with there style.   
2                                             None   
3                                             None   
4                                             None   

                                      text_no_marker            label  \
0  A drawer is usually found in a piece of furnit...       stereotype   
1          Hairdressers are unique with there style.  anti-stereotype   
2   She is a county judge. Some sentences are short.        unrelated   
3  I don't know why I need a broker. I don't need...       stereotype   
4  Soon I have to go see my physician. He's young...  anti-stereotype   

  stereotype_type           binary_class                 multi_class  \
0      profession  stereotype_profession       stereotype_profession   
1      profession              unrelated  anti-stereotype_profession   
2      profession   

In [23]:
from stereotype_def import stereotype_definition
import random

sampled_rows = sample_mgsd.sample(n=5, random_state=42).reset_index(drop=True)

tot = TreeOfThought(client, model=model, max_branching_factor=3, max_depth=3, stereotype_definition=stereotype_definition)

def label_predicted(label: str):
    if label.lower() == "yes":
        prediction = "stereotype"
    elif label.lower() == "no":
        prediction = "anti-stereotype"
    else:
        prediction = "unrelated" 

    return prediction


for idx, row in sampled_rows.iterrows():
    print(f"\n===========================")
    print(f"🧪 Example {idx + 1}")
    print(f"===========================\n")

    text = row["text_no_marker"]
    true_label = row["label"].strip().lower()

    print(f"📜 Input Text:\n{text}\n")

    tot.total_tokens = 0
    tot.total_prompt_tokens = 0
    tot.total_completion_tokens = 0
    tot.total_latency = 0.0
    tot.total_calls = 0

    solution = tot.solve(text)
    path_vote = tot._get_majority_vote_from_path(solution)
    tree_vote = tot._get_majority_vote_from_tree()
    leaf_vote = tot._get_majority_vote_from_leafs()
    
    prediction_path = label_predicted(path_vote)
    prediction_tree = label_predicted(tree_vote)
    prediction_leaf = label_predicted(leaf_vote)

    w_tree_vote = tot._get_majority_vote_from_tree(weighted=True)
    w_path_vote = tot._get_majority_vote_from_path(solution, weighted=True)
    w_leaf_vote = tot._get_majority_vote_from_leafs(weighted=True)

    w_prediction_path = label_predicted(w_path_vote)
    w_prediction_tree = label_predicted(w_tree_vote)
    w_prediction_leaf = label_predicted(w_leaf_vote)

    print(f"🌲 Tree Prediction     : {prediction_tree}  → Correct? {prediction_tree == true_label}")
    print(f"🪜 Best Path Prediction: {prediction_path}  → Correct? {prediction_path == true_label}")
    print(f"🍃 Leaf Vote Prediction : {prediction_leaf}   → Correct? {prediction_leaf == true_label}")

    print("\n\n⚖️  Weighted Voting Results")
    print(f"🌲 Weighted Tree Prediction     : {w_prediction_tree}   → Correct? {w_prediction_tree == true_label}")
    print(f"🪜 Weighted Best Path Prediction: {w_prediction_path}   → Correct? {w_prediction_path == true_label}")
    print(f"🍃 Weighted Leaf Vote Prediction: {w_prediction_leaf}   → Correct? {w_prediction_leaf == true_label}")

    print("     True label was :", true_label)

    print("\n")
    print("\n🧠 Full Tree Structure:\n")
    tot.print_full_tree()
    print("\n")
    
    print("\n📊 LLM Metrics:")
    print(f"  Total LLM Calls        : {tot.total_calls}")
    print(f"  Total Tokens Used      : {tot.total_tokens}")
    print(f"       Prompt Tokens     : {tot.total_prompt_tokens}")
    print(f"       Completion Tokens : {tot.total_completion_tokens}")
    if tot.total_calls > 0:
        avg_tokens = tot.total_tokens / tot.total_calls
        print(f"  Avg Tokens per Call    : {avg_tokens:.2f}")


🧪 Example 1

📜 Input Text:
A drawer is usually found in a piece of furniture, such as a nightstand, table, chest, or dresser. They seem to get stuck when you want to use them the most.

🌲 Tree Prediction     : anti-stereotype  → Correct? False
🪜 Best Path Prediction: anti-stereotype  → Correct? False
🍃 Leaf Vote Prediction : anti-stereotype   → Correct? False


⚖️  Weighted Voting Results
🌲 Weighted Tree Prediction     : anti-stereotype   → Correct? False
🪜 Weighted Best Path Prediction: anti-stereotype   → Correct? False
🍃 Weighted Leaf Vote Prediction: anti-stereotype   → Correct? False
     True label was : stereotype



🧠 Full Tree Structure:

🧠 Thought ID 0
→ Score: 0.00
→ Content: A drawer is usually found in a piece of furniture, such as a nightstand, table, chest, or dresser. They seem to get stuck when you want to use them the most.
------------------------------------------------------------
    🧠 Thought ID 0.1
    → Score: 0.60
    → Content: Drawers often get stuck due to